# Notebook 00 — Seed Audit (Read-Only)

## Purpose

Before building the multi-seed runner, scan every notebook in the X-IDS pipeline to find:

1. Every line that mentions `seed`, `random_state`, or RNG initialization
2. Every hardcoded file path that would collide if we run the pipeline at a different seed
3. Every numpy save/load that needs seed-suffixed naming

The output is a checklist the multi-seed runner needs to handle.

## What this notebook does NOT do

- Does not modify any notebooks
- Does not run any pipeline stages
- Does not create any model/calibration outputs

Read-only inventory. Output is one CSV + one markdown summary.

## Scope

Notebooks audited (the seed-dependent stages):
- 02_* (training)
- 03_* (calibration), 03d, 03e
- 04_* (SHAP), 04b, 04c
- 05_* (stability), 05c
- 06_* (Krishna)
- 07c, 07d, 07e, 07f (SCTS / health flag / Phase A)

NOT audited (seed-independent or already done):
- 08* (Phase B — fixed thresholds, no seed dependency)
- Any docs/ files


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)
print(f'Ready: {os.getcwd()}')

Mounted at /content/drive
Ready: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import nbformat as nbf
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

NOTEBOOKS_TO_AUDIT = [
    '02_data_pipeline.ipynb',
    '02b_unsw_pipeline.ipynb',
    '02c_cic_pipeline.ipynb',
    '03_nsl_calibration_v2.ipynb',
    '03_unsw_calibration.ipynb',
    '03_cic_calibration.ipynb',
    '03d_calibration_bootstrap_cis.ipynb',
    '03e_refit_hybrid_calibrators.ipynb',
    '04_shap_analysis.ipynb',
    '04b_calibration_shap_validation.ipynb',
    '04c_shap_canonical.ipynb',
    '05_stability_analysis.ipynb',
    '05c_stability_canonical.ipynb',
    '06_krishna_agreement_v3.ipynb',
    '07c_scts_canonical_mondrian_v2.ipynb',
    '07d_scts_calib_health.ipynb',
    '07e_phase_a_strict_protocol.ipynb',
    '07f_phase_a_diagnostic.ipynb',
]

# Patterns to match (case-sensitive where it matters)
SEED_PATTERNS = [
    (r'\bSEED\s*=\s*\d+', 'SEED_assignment'),
    (r'\bseed\s*=\s*\d+', 'seed_kwarg'),
    (r'random_state\s*=\s*\d+', 'sklearn_random_state'),
    (r'random_state\s*=\s*\w+', 'sklearn_random_state_var'),
    (r'np\.random\.seed\(', 'np_random_seed'),
    (r'np\.random\.RandomState\(', 'np_random_RandomState'),
    (r'tf\.random\.set_seed\(', 'tf_random_set_seed'),
    (r'tf\.keras\.utils\.set_random_seed\(', 'tf_keras_set_random_seed'),
    (r'torch\.manual_seed\(', 'torch_manual_seed'),
    (r'random\.seed\(', 'python_random_seed'),
]

# Patterns for file paths that might collide
PATH_PATTERNS = [
    (r'(models/[a-z_0-9]+/[a-z_0-9]+\.npy)', 'model_output_path'),
    (r'(models/[a-z_0-9]+/[a-z_0-9]+\.joblib)', 'model_pickle_path'),
    (r'(calibrators/[a-z_0-9]+/[a-z_0-9_]+\.joblib)', 'calibrator_path'),
    (r'(shap_values/[a-z_0-9]+/[a-z_0-9_]+\.npy)', 'shap_output_path'),
    (r'(results/tables/[a-z_0-9_]+\.csv)', 'results_csv_path'),
    (r'(results/tables/[a-z_0-9_]+\.json)', 'results_json_path'),
]

print(f'Will audit {len(NOTEBOOKS_TO_AUDIT)} notebooks')
print(f'Seed patterns: {len(SEED_PATTERNS)}')
print(f'Path patterns: {len(PATH_PATTERNS)}')

Will audit 18 notebooks
Seed patterns: 10
Path patterns: 6


In [3]:
# Scan each notebook
nb_dir = Path(REPO) / 'notebooks'
seed_findings = []
path_findings = []
missing_notebooks = []

for nb_name in NOTEBOOKS_TO_AUDIT:
    nb_path = nb_dir / nb_name
    if not nb_path.exists():
        missing_notebooks.append(nb_name)
        continue

    try:
        nb_obj = nbf.read(str(nb_path), as_version=4)
    except Exception as e:
        print(f'Failed to read {nb_name}: {e}')
        continue

    for cell_idx, cell in enumerate(nb_obj.cells):
        if cell.cell_type != 'code':
            continue
        src = cell.source if isinstance(cell.source, str) else '\n'.join(cell.source)
        for line_idx, line in enumerate(src.split('\n')):
            line_stripped = line.strip()
            if line_stripped.startswith('#'):  # skip comments-only lines
                continue

            for pat, kind in SEED_PATTERNS:
                if re.search(pat, line):
                    seed_findings.append({
                        'notebook': nb_name,
                        'cell_idx': cell_idx,
                        'line_idx': line_idx,
                        'pattern': kind,
                        'line': line.strip()[:200],
                    })

            for pat, kind in PATH_PATTERNS:
                m = re.search(pat, line)
                if m:
                    path_findings.append({
                        'notebook': nb_name,
                        'cell_idx': cell_idx,
                        'line_idx': line_idx,
                        'pattern': kind,
                        'path_matched': m.group(1),
                        'line': line.strip()[:200],
                    })

print(f'Notebooks scanned: {len(NOTEBOOKS_TO_AUDIT) - len(missing_notebooks)}')
print(f'Missing notebooks: {missing_notebooks}')
print(f'Seed-related findings: {len(seed_findings)}')
print(f'Path-related findings: {len(path_findings)}')

df_seed = pd.DataFrame(seed_findings)
df_path = pd.DataFrame(path_findings)

Notebooks scanned: 14
Missing notebooks: ['02_data_pipeline.ipynb', '02b_unsw_pipeline.ipynb', '02c_cic_pipeline.ipynb', '05_stability_analysis.ipynb']
Seed-related findings: 39
Path-related findings: 60


In [4]:
# Per-notebook summary
print('=' * 80)
print('PER-NOTEBOOK SEED FINDINGS SUMMARY')
print('=' * 80)

if len(df_seed) > 0:
    summary = df_seed.groupby(['notebook', 'pattern']).size().unstack(fill_value=0)
    print(summary.to_string())
else:
    print('No seed findings.')

print()
print('=' * 80)
print('PATH FINDINGS SUMMARY (top patterns per notebook)')
print('=' * 80)
if len(df_path) > 0:
    summary_p = df_path.groupby(['notebook', 'pattern']).size().unstack(fill_value=0)
    print(summary_p.to_string())
else:
    print('No path findings.')

PER-NOTEBOOK SEED FINDINGS SUMMARY
pattern                                SEED_assignment  np_random_RandomState  np_random_seed  python_random_seed  sklearn_random_state  sklearn_random_state_var  torch_manual_seed
notebook                                                                                                                                                                            
03_cic_calibration.ipynb                             1                      0               1                   1                     0                         1                  0
03_nsl_calibration_v2.ipynb                          1                      0               1                   1                     0                         0                  0
03_unsw_calibration.ipynb                            0                      0               0                   0                     1                         1                  0
03d_calibration_bootstrap_cis.ipynb                  1      

In [5]:
# Show the actual seed VALUES used — most will be 42, some might be different
print('=' * 80)
print('SEED VALUES FOUND (extract the literal numbers)')
print('=' * 80)

seed_value_pattern = re.compile(r'(?:SEED|seed|random_state|\.seed)\s*[(=]\s*(\d+)')
value_findings = []
for _, row in df_seed.iterrows():
    matches = seed_value_pattern.findall(row['line'])
    for v in matches:
        value_findings.append({
            'notebook': row['notebook'],
            'pattern': row['pattern'],
            'value': int(v),
            'line': row['line'],
        })

if value_findings:
    df_values = pd.DataFrame(value_findings)
    print(f'\nTotal seed-value instances: {len(df_values)}')
    print(f'\nDistinct values used:')
    print(df_values['value'].value_counts().to_string())
    print(f'\nPer-notebook distinct values:')
    for nb_name, sub in df_values.groupby('notebook'):
        vals = sorted(sub['value'].unique())
        print(f'  {nb_name}: {vals}')
else:
    print('No literal seed values extracted.')

SEED VALUES FOUND (extract the literal numbers)

Total seed-value instances: 13

Distinct values used:
value
42    13

Per-notebook distinct values:
  03_cic_calibration.ipynb: [np.int64(42)]
  03_nsl_calibration_v2.ipynb: [np.int64(42)]
  03_unsw_calibration.ipynb: [np.int64(42)]
  03d_calibration_bootstrap_cis.ipynb: [np.int64(42)]
  04_shap_analysis.ipynb: [np.int64(42)]
  04b_calibration_shap_validation.ipynb: [np.int64(42)]
  04c_shap_canonical.ipynb: [np.int64(42)]
  05c_stability_canonical.ipynb: [np.int64(42)]
  06_krishna_agreement_v3.ipynb: [np.int64(42)]
  07c_scts_canonical_mondrian_v2.ipynb: [np.int64(42)]
  07e_phase_a_strict_protocol.ipynb: [np.int64(42)]
  07f_phase_a_diagnostic.ipynb: [np.int64(42)]


In [6]:
# Look for uses of SEED as variable (e.g., random_state=SEED) — these are GOOD; they parameterize
print('=' * 80)
print('GOOD: lines using SEED as variable (already parameterized)')
print('=' * 80)

if len(df_seed) > 0:
    parameterized = df_seed[df_seed['line'].str.contains(r'(?:random_state|seed)\s*=\s*SEED', regex=True)]
    print(f'\nFound {len(parameterized)} lines using SEED variable:')
    if len(parameterized) > 0:
        for _, row in parameterized.iterrows():
            print(f'  {row["notebook"]} cell {row["cell_idx"]}: {row["line"][:100]}')

print()
print('=' * 80)
print('BAD: lines with hardcoded numeric seed (need parameterizing)')
print('=' * 80)

if len(df_seed) > 0:
    hardcoded = df_seed[df_seed['line'].str.contains(r'(?:random_state|seed)\s*=\s*\d+', regex=True)]
    print(f'\nFound {len(hardcoded)} lines with hardcoded numeric seed:')
    for _, row in hardcoded.iterrows():
        print(f'  {row["notebook"]} cell {row["cell_idx"]} line {row["line_idx"]}: {row["line"][:120]}')

GOOD: lines using SEED as variable (already parameterized)

Found 1 lines using SEED variable:
  03_cic_calibration.ipynb cell 4: all_idx, test_size=0.50, stratify=y_test_5, random_state=SEED

BAD: lines with hardcoded numeric seed (need parameterizing)

Found 2 lines with hardcoded numeric seed:
  03_unsw_calibration.ipynb cell 7 line 9: random_state=42,
  03_unsw_calibration.ipynb cell 7 line 9: random_state=42,


In [7]:
# Which paths write (vs read) — write paths are the ones we need to seed-suffix
print('=' * 80)
print('PATH WRITES vs READS — which files need seed-suffixed naming')
print('=' * 80)

if len(df_path) > 0:
    # Identify writes (np.save, joblib.dump, .to_csv, json.dump etc.)
    write_indicators = ['np.save', '.dump(', '.to_csv(', 'json.dump', 'open(', 'savefig']

    is_write = df_path['line'].apply(lambda L: any(ind in L for ind in write_indicators))
    df_path['is_write'] = is_write

    writes = df_path[df_path['is_write']]
    reads = df_path[~df_path['is_write']]

    print(f'\nTotal path references: {len(df_path)}')
    print(f'  Writes (need seed-suffix): {len(writes)}')
    print(f'  Reads (point to existing files): {len(reads)}')

    print(f'\nUnique writable paths to seed-suffix:')
    unique_writes = writes['path_matched'].drop_duplicates().tolist()
    for p in unique_writes[:40]:  # cap output
        print(f'  {p}')
    if len(unique_writes) > 40:
        print(f'  ... and {len(unique_writes) - 40} more')

PATH WRITES vs READS — which files need seed-suffixed naming

Total path references: 60
  Writes (need seed-suffix): 1
  Reads (point to existing files): 59

Unique writable paths to seed-suffix:
  results/tables/calibration_brier_recomputed.csv


In [8]:
# Save the full audit
out_dir = Path(REPO) / 'docs'
out_dir.mkdir(parents=True, exist_ok=True)

if len(df_seed) > 0:
    df_seed.to_csv(out_dir / 'seed_audit_seed_findings.csv', index=False)
    print(f'Saved: docs/seed_audit_seed_findings.csv ({len(df_seed)} rows)')

if len(df_path) > 0:
    df_path.to_csv(out_dir / 'seed_audit_path_findings.csv', index=False)
    print(f'Saved: docs/seed_audit_path_findings.csv ({len(df_path)} rows)')

# Write a summary markdown
summary_md = f"""# Seed Audit Summary

**Date**: {datetime.now().strftime('%Y-%m-%d')}
**Purpose**: pre-flight checklist for the multi-seed runner

## Scope

Notebooks audited: {len(NOTEBOOKS_TO_AUDIT) - len(missing_notebooks)}
Missing notebooks: {missing_notebooks if missing_notebooks else 'none'}

## Findings

- Seed/RNG references: {len(df_seed)}
- File path references: {len(df_path)}

## What the multi-seed runner needs to do

1. **Parameterize all SEED references**: replace every literal `random_state=42`, `seed=42`, `np.random.seed(42)` with the loop variable.
2. **Seed-suffix all writable output paths**: model files, calibrators, SHAP arrays, results tables/JSONs.
3. **Reuse canonical 1000 across seeds**: do NOT re-derive `canonical_eval_idx` per seed.
4. **Phase B not rerun**: 08* notebooks are seed-independent at the threshold-search level.

## Detailed CSVs

- `docs/seed_audit_seed_findings.csv` — every seed/RNG mention with notebook + cell + line context
- `docs/seed_audit_path_findings.csv` — every file-path reference with read/write classification

## Next step

Use these findings to design `00_multi_seed_runner.ipynb`. The runner will:
- Define each pipeline stage as an inline function with explicit seed parameter
- Save outputs to `models/{{ds}}/seed{{N}}/` instead of `models/{{ds}}/`
- Skip stages whose outputs already exist (resumability)
- Run all 9 stages per seed for SEEDS = [123, 456, 789]
"""

with open(out_dir / 'seed_audit_summary.md', 'w') as f:
    f.write(summary_md)
print(f'Saved: docs/seed_audit_summary.md ({len(summary_md)} chars)')
print()
print('Done. Audit complete, no modifications made.')

Saved: docs/seed_audit_seed_findings.csv (39 rows)
Saved: docs/seed_audit_path_findings.csv (60 rows)
Saved: docs/seed_audit_summary.md (1370 chars)

Done. Audit complete, no modifications made.


In [9]:
from pathlib import Path
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
nb_dir = Path(REPO) / 'notebooks'

print('=== All notebooks in notebooks/ ===')
for p in sorted(nb_dir.glob('*.ipynb')):
    print(f'  {p.name} ({p.stat().st_size:,} bytes)')

print()
print('=== Notebooks containing model training (RF/XGB/DNN fit) ===')
import nbformat as nbf
keywords = ['RandomForestClassifier', 'XGBClassifier', 'model.fit(', 'Sequential(', 'compile(', 'tf.keras.Model']
for p in sorted(nb_dir.glob('*.ipynb')):
    try:
        nb = nbf.read(str(p), as_version=4)
    except Exception as e:
        continue
    text = '\n'.join(c.source for c in nb.cells if c.cell_type == 'code' and isinstance(c.source, str))
    hits = [k for k in keywords if k in text]
    if hits:
        print(f'  {p.name}: {hits}')

=== All notebooks in notebooks/ ===
  00_seed_audit.ipynb (22,928 bytes)
  01_cic_data_exploration.ipynb (16,683 bytes)
  01_cic_data_exploration_v2.ipynb (76,689 bytes)
  01_data_exploration.ipynb (104,969 bytes)
  01_data_exploration_v2.ipynb (74,393 bytes)
  01_unsw_data_exploration.ipynb (100,120 bytes)
  01_unsw_data_exploration_v2.ipynb (82,556 bytes)
  01b_unsw_10class_sensitivity.ipynb (20,554 bytes)
  02_cic_train_models.ipynb (21,133 bytes)
  02_cic_train_models_v2.ipynb (38,668 bytes)
  02_nsl_rf_retrain_v2.ipynb (19,101 bytes)
  02_train_models_v2.ipynb (34,813 bytes)
  02_unsw_train_models.ipynb (33,091 bytes)
  02_unsw_train_models_v2.ipynb (41,435 bytes)
  02b_rare_class_boost.ipynb (78,015 bytes)
  02b_unsw_dnn_diagnostic.ipynb (174,779 bytes)
  02c_unsw_dnn_retrain.ipynb (22,730 bytes)
  03_calibration.ipynb (226,497 bytes)
  03_calibration_v2.ipynb (281,680 bytes)
  03_cic_calibration.ipynb (12,586 bytes)
  03_cic_calibration_v2.ipynb (40,780 bytes)
  03_nsl_calibrati

In [10]:
import nbformat as nbf
from pathlib import Path
import re

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
nb_dir = Path(REPO) / 'notebooks'

notebooks_of_interest = [
    '03_nsl_calibration_v2.ipynb',
    '03_unsw_calibration.ipynb',
    '03d_calibration_bootstrap_cis.ipynb',
    '04c_shap_canonical.ipynb',
    '05c_stability_canonical.ipynb',
    '06_krishna_agreement_v3.ipynb',
    '07c_scts_canonical_mondrian_v2.ipynb',
    '07d_scts_calib_health.ipynb',
    '07e_phase_a_strict_protocol.ipynb',
    '07f_phase_a_diagnostic.ipynb',
]

for nb_name in notebooks_of_interest:
    p = nb_dir / nb_name
    if not p.exists():
        print(f'{nb_name}: NOT FOUND'); continue
    nb = nbf.read(str(p), as_version=4)
    text_lines = []
    for ci, c in enumerate(nb.cells):
        if c.cell_type != 'code': continue
        for li, line in enumerate(c.source.split('\n')):
            if re.search(r'\b(SEED|seed|random_state)\s*=', line) or 'np.random' in line or 'torch.manual_seed' in line or 'tf.random' in line:
                text_lines.append((ci, li, line.strip()))
    print(f'\n=== {nb_name} ({len(text_lines)} matches) ===')
    for ci, li, line in text_lines[:8]:
        print(f'  cell {ci} line {li}: {line[:140]}')
    if len(text_lines) > 8:
        print(f'  ... +{len(text_lines)-8} more')


=== 03_nsl_calibration_v2.ipynb (2 matches) ===
  cell 3 line 14: SEED = 42
  cell 3 line 15: np.random.seed(SEED)

=== 03_unsw_calibration.ipynb (1 matches) ===
  cell 7 line 9: random_state=42,

=== 03d_calibration_bootstrap_cis.ipynb (3 matches) ===
  cell 3 line 10: SEED = 42
  cell 3 line 11: rng = np.random.default_rng(SEED)
  cell 7 line 41: rng_local = np.random.default_rng(SEED)

=== 04c_shap_canonical.ipynb (2 matches) ===
  cell 3 line 17: SEED = 42  # Canonical seed for all dataset sample selections
  cell 7 line 9: rng = np.random.default_rng(SEED)

=== 05c_stability_canonical.ipynb (8 matches) ===
  cell 3 line 19: SEED = 42
  cell 3 line 20: np.random.seed(SEED)
  cell 3 line 21: torch.manual_seed(SEED)
  cell 7 line 2: rng = np.random.RandomState(seed)
  cell 7 line 16: def pgd_attack(model_raw, X, y, epsilon, alpha, steps, seed=None):
  cell 7 line 22: torch.manual_seed(seed)
  cell 9 line 22: X_gauss = gaussian_perturbation(X_canonical, EPSILON, seed=SEED)
  cell 9 l

In [11]:
import nbformat as nbf
from pathlib import Path
import re

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
nb_dir = Path(REPO) / 'notebooks'

# The three unaudited training notebooks + the v2 calibration variants we missed
TARGETS = [
    '02_train_models_v2.ipynb',           # NSL training
    '02_unsw_train_models_v2.ipynb',      # UNSW training
    '02_cic_train_models_v2.ipynb',       # CIC training
    '03_unsw_calibration_v2.ipynb',       # UNSW calibration (v2 form)
    '03_cic_calibration_v2.ipynb',        # CIC calibration (v2 form)
    '08_bootstrap_cis.ipynb',             # Bootstrap CIs
]

for nb_name in TARGETS:
    p = nb_dir / nb_name
    if not p.exists():
        print(f'\n=== {nb_name}: NOT FOUND ===')
        continue
    nb = nbf.read(str(p), as_version=4)
    print(f'\n=== {nb_name} ===')
    lines_of_interest = []
    save_lines = []
    for ci, c in enumerate(nb.cells):
        if c.cell_type != 'code': continue
        src = c.source if isinstance(c.source, str) else '\n'.join(c.source)
        for li, line in enumerate(src.split('\n')):
            # Seed/RNG lines
            if re.search(r'\b(SEED|seed|random_state)\s*=', line) or 'np.random' in line or 'torch.manual_seed' in line or 'tf.random' in line or 'tf.keras.utils.set_random_seed' in line:
                lines_of_interest.append((ci, li, line.strip()))
            # File-write lines (save outputs)
            if any(ind in line for ind in ['np.save(', 'joblib.dump(', '.to_csv(', '.save(', 'json.dump(']):
                save_lines.append((ci, li, line.strip()))
    print(f'  Seed/RNG lines: {len(lines_of_interest)}')
    for ci, li, line in lines_of_interest[:15]:
        print(f'    cell {ci} line {li}: {line[:140]}')
    if len(lines_of_interest) > 15:
        print(f'    +{len(lines_of_interest)-15} more')
    print(f'  Save lines: {len(save_lines)}')
    for ci, li, line in save_lines[:10]:
        print(f'    cell {ci} line {li}: {line[:140]}')
    if len(save_lines) > 10:
        print(f'    +{len(save_lines)-10} more')


=== 02_train_models_v2.ipynb ===
  Seed/RNG lines: 5
    cell 3 line 16: SEED = 42
    cell 3 line 17: np.random.seed(SEED); torch.manual_seed(SEED)
    cell 6 line 1: RF_HP = dict(n_estimators=200, n_jobs=-1, random_state=SEED)
    cell 6 line 5: random_state=SEED, n_jobs=-1, eval_metric='mlogloss',
    cell 12 line 5: smote = SMOTE(random_state=SEED, k_neighbors=k_neighbors)
  Save lines: 8
    cell 6 line 41: np.save(PREDS_DIR / f'{name}_test_pred.npy',  test_pred)
    cell 6 line 42: np.save(PREDS_DIR / f'{name}_test_proba.npy', test_proba)
    cell 6 line 43: np.save(PREDS_DIR / f'{name}_calib_pred.npy',  calib_pred)
    cell 6 line 44: np.save(PREDS_DIR / f'{name}_calib_proba.npy', calib_proba)
    cell 8 line 68: torch.save({
    cell 17 line 16: df.to_csv(TABLES_DIR / 'nslkdd_v2_model_comparison.csv', index=False)
    cell 19 line 19: ablation_df.to_csv(TABLES_DIR / 'nslkdd_v2_imbalance_ablation.csv', index=False)
    cell 21 line 1: json.dump(ALL_METRICS, f, indent=2)

=== 02

In [12]:
import nbformat as nbf
from pathlib import Path
import re

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
nb_dir = Path(REPO) / 'notebooks'

# Verify the three remaining notebooks I haven't seen seed-handling for
TARGETS = ['03e_refit_hybrid_calibrators.ipynb', '07d_scts_calib_health.ipynb']

for nb_name in TARGETS:
    p = nb_dir / nb_name
    if not p.exists():
        print(f'\n=== {nb_name}: NOT FOUND ==='); continue
    nb = nbf.read(str(p), as_version=4)
    print(f'\n=== {nb_name} ===')
    for ci, c in enumerate(nb.cells):
        if c.cell_type != 'code': continue
        src = c.source if isinstance(c.source, str) else '\n'.join(c.source)
        for li, line in enumerate(src.split('\n')):
            if re.search(r'\b(SEED|seed|random_state)\s*=', line) or 'np.random' in line or 'torch.manual_seed' in line:
                print(f'  cell {ci} line {li}: {line.strip()[:140]}')

    # Also check for path-related output writes
    print(f'  --- Save lines ---')
    for ci, c in enumerate(nb.cells):
        if c.cell_type != 'code': continue
        src = c.source if isinstance(c.source, str) else '\n'.join(c.source)
        for li, line in enumerate(src.split('\n')):
            if any(ind in line for ind in ['np.save(', 'joblib.dump(', '.to_csv(', '.save(', 'json.dump(']):
                print(f'    cell {ci} line {li}: {line.strip()[:140]}')


=== 03e_refit_hybrid_calibrators.ipynb ===
  --- Save lines ---
    cell 11 line 18: joblib.dump(to_save, out_path)

=== 07d_scts_calib_health.ipynb ===
  --- Save lines ---
    cell 11 line 16: df_health.to_csv(out_path, index=False)
    cell 13 line 18: df_scts.to_csv(out_path, index=False)
    cell 17 line 54: json.dump(to_json_safe(summary), f, indent=2, default=str)
    cell 20 line 52: json.dump(summary, f, indent=2)
